# Notebook 04 — XGBoost Training
**Week 2 Task:** Train XGBoost signal classifier on all 49 Nifty 50 stocks.

Target: accuracy > 57% with balanced UP/DOWN recall (not predicting one class always).

Split: Train 2000–2020 | Val 2021 (inside XGBoost early stopping) | Test 2022–2026

In [ ]:
import os
os.chdir(r'C:\Users\Aryan\Desktop\everything\PROJECTS\200%BOT\ai-trading-bot\ai-trading-bot')
print('Working dir:', os.getcwd())

In [ ]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.data.preprocess import preprocess_symbol, get_feature_columns, get_train_test_split
from src.models.xgboost_model import XGBoostModel

print('Imports OK')

## Step 1 — Train on RELIANCE first (single stock test)

In [ ]:
df = preprocess_symbol('RELIANCE')
feat_cols = get_feature_columns(df)
train_df, test_df = get_train_test_split(df, test_start='2022-01-01')  # CHANGED: was 2019-01-01

print(f'Features: {len(feat_cols)}')
print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(f'Train period: {train_df.index[0].date()} to {train_df.index[-1].date()}')
print(f'Test period:  {test_df.index[0].date()} to {test_df.index[-1].date()}')

In [ ]:
model = XGBoostModel()
model.train(train_df, feat_cols, val_df=test_df)
results = model.evaluate(test_df, feat_cols)

print('\nTest Results:')
for k, v in results.items():
    print(f'  {k}: {v}')

if results['accuracy'] >= 0.55:
    print('\n✅ Above 55% target')
else:
    print('\n⚠️ Below 55% — check features')

In [ ]:
# Feature importance — what does XGBoost actually use?
top = model.top_features(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
ax.set_title('Top 20 Feature Importances — RELIANCE XGBoost')
ax.set_xlabel('Importance Score')
plt.tight_layout()
os.makedirs('models/results', exist_ok=True)
plt.savefig('models/results/xgb_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print(top.to_string(index=False))

## Step 2 — Walk-forward cross validation (more reliable than single split)

In [ ]:
print('Running 5-fold walk-forward CV on RELIANCE...')
cv_results = model.cross_validate(df, feat_cols, n_splits=5)
print(f"\nCV Results:")
for k, v in cv_results.items():
    print(f'  {k}: {v}')

if cv_results['mean_accuracy'] >= 0.55:
    print('\n✅ CV accuracy above 55% — model is robust')
else:
    print('\n⚠️ CV accuracy below 55% — model may be overfitting single split')

## Step 3 — Train on ALL stocks and rank by accuracy

In [ ]:
results_all = []
failed = []

raw_files = [f.replace('.csv','') for f in os.listdir('data/raw')
             if f.endswith('.csv') and not f.endswith('.NS.csv')]
skip = {'INFRATEL', 'NIFTY50_all', 'stock_metadata'}
symbols = [s for s in raw_files if s not in skip]

print(f'Training XGBoost on {len(symbols)} symbols...')
print('Takes ~2 minutes total\n')

for i, symbol in enumerate(symbols, 1):
    try:
        df_s = preprocess_symbol(symbol)
        if df_s is None or len(df_s) < 400:
            continue
        fc = get_feature_columns(df_s)
        tr, te = get_train_test_split(df_s, test_start='2022-01-01')  # CHANGED: was 2019-01-01
        if len(te) < 100:
            continue
        m = XGBoostModel()
        m.train(tr, fc, val_df=te)
        r = m.evaluate(te, fc)
        r['symbol'] = symbol
        results_all.append(r)
        print(f'[{i}/{len(symbols)}] {symbol}: acc={r["accuracy"]:.3f} | recall_down={r["recall_down"]:.2f} | recall_up={r["recall_up"]:.2f}')
    except Exception as e:
        print(f'[{i}/{len(symbols)}] {symbol}: ERROR — {e}')
        failed.append(symbol)

print(f'\nDone: {len(results_all)} trained | {len(failed)} failed')

In [ ]:
results_df = pd.DataFrame(results_all).sort_values('accuracy', ascending=False)

print('All stocks by accuracy:')
print(results_df[['symbol','accuracy','recall_down','recall_up','f1_up']].to_string(index=False))
print()
print(f'Mean accuracy: {results_df["accuracy"].mean():.3f}')
print(f'Stocks above 55%: {(results_df["accuracy"] >= 0.55).sum()}')
print(f'Stocks above 57%: {(results_df["accuracy"] >= 0.57).sum()}')

# Filter: only keep stocks with BALANCED recall (both UP and DOWN recall > 40%)
# This ensures model isn't just predicting one direction always
balanced = results_df[
    (results_df['recall_down'] > 0.40) &
    (results_df['recall_up'] > 0.40)
]
print(f'\nBalanced stocks (both recall > 40%): {len(balanced)}')
print(balanced[['symbol','accuracy','recall_down','recall_up']].to_string(index=False))

results_df.to_csv('models/results/xgb_all_stocks.csv', index=False)

## Step 4 — Train final model on all stocks combined

In [ ]:
# Use top 15 balanced stocks for final model
top_symbols = balanced.head(15)['symbol'].tolist() if len(balanced) >= 5 else results_df.head(15)['symbol'].tolist()
print(f'Training final model on: {top_symbols}\n')

combined_train, combined_test = [], []
for symbol in top_symbols:
    df_s = preprocess_symbol(symbol)
    if df_s is not None:
        fc = get_feature_columns(df_s)
        tr, te = get_train_test_split(df_s, test_start='2022-01-01')  # CHANGED: was 2019-01-01
        combined_train.append(tr)
        combined_test.append(te)

train_all = pd.concat(combined_train).sort_index()
test_all = pd.concat(combined_test).sort_index()
print(f'Combined train: {len(train_all)} | test: {len(test_all)}')

In [ ]:
final_model = XGBoostModel()
final_model.train(train_all, fc, val_df=test_all)
final_results = final_model.evaluate(test_all, fc)

print('\nFinal XGBoost Results:')
for k, v in final_results.items():
    print(f'  {k}: {v}')

final_model.save('xgboost')
print('\n✅ Final XGBoost saved to models/saved/xgboost.pkl')

In [ ]:
# Final feature importance plot
top_final = final_model.top_features(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2196F3' if i < 5 else '#64B5F6' if i < 10 else '#BBDEFB' for i in range(len(top_final))]
ax.barh(top_final['feature'][::-1], top_final['importance'][::-1], color=colors[::-1])
ax.set_title('Top 20 Features — Final XGBoost Model (all stocks)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('models/results/xgb_final_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 10 most predictive features:')
print(top_final.head(10).to_string(index=False))

## ✅ Done

XGBoost is trained and saved. Check `models/results/xgb_all_stocks.csv` for full results.

Next: open `05_ensemble_backtest.ipynb` to combine LSTM + XGBoost into the ensemble.